# LuxeForLess VTO on Kaggle GPU

1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → ON**
3. Kaggle **Secrets**: `NGROK_AUTHTOKEN` from [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)
4. GitHub repo must be **public** (or add optional secret `GITHUB_TOKEN` for private repos)

When done, copy the **ngrok HTTPS URL** into Vercel → `NEXT_PUBLIC_VTO_SERVICE_URL`

> **Note:** Kaggle does NOT auto-update from GitHub. Re-run this notebook after code changes. Vercel auto-deploys only if GitHub is connected in Vercel dashboard.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["NGROK_AUTHTOKEN"] = secrets.get_secret("NGROK_AUTHTOKEN")
os.environ["LUXEFORLESS_REPO_URL"] = "https://github.com/mintukumar0000/luxeforless.git"
os.environ["VTO_NUM_TIMESTEPS"] = "4"

print("NGROK token loaded:", "yes" if os.environ["NGROK_AUTHTOKEN"] else "NO — add NGROK_AUTHTOKEN in Add-ons → Secrets")
print("Repo:", os.environ["LUXEFORLESS_REPO_URL"])

In [ ]:
import os
import shutil
import subprocess
import sys

# Self-contained: works even if Cell 1 wasn't run (e.g. after session restart)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ.setdefault("NGROK_AUTHTOKEN", secrets.get_secret("NGROK_AUTHTOKEN"))
except Exception:
    pass

os.environ.setdefault("LUXEFORLESS_REPO_URL", "https://github.com/mintukumar0000/luxeforless.git")
os.environ.setdefault("VTO_NUM_TIMESTEPS", "4")

if not os.environ.get("NGROK_AUTHTOKEN"):
    raise RuntimeError("NGROK_AUTHTOKEN missing — add it in Add-ons → Secrets, then run this cell again")

work = "/kaggle/working/luxeforless"
script = f"{work}/deploy/kaggle/run_vto.py"
repo = os.environ["LUXEFORLESS_REPO_URL"]

# Optional: private repo support via Kaggle secret GITHUB_TOKEN
try:
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if token and "github.com" in repo and "@" not in repo:
        repo = repo.replace("https://", f"https://{token}@")
        print("Using GITHUB_TOKEN for private repo clone")
except Exception:
    pass

if os.path.exists(work) and not os.path.exists(script):
    print("Removing incomplete clone...")
    shutil.rmtree(work)

if not os.path.exists(script):
    print("Cloning luxeforless repo...")
    subprocess.check_call(["git", "clone", "--depth", "1", repo, work])

print("NGROK token loaded: yes")
print("Repo:", repo)
print("Starting VTO service (first run downloads ~2GB weights — allow 10-15 min)...")
print("When you see the ngrok HTTPS URL below, paste it into Vercel as NEXT_PUBLIC_VTO_SERVICE_URL\n")

subprocess.check_call([sys.executable, script])